# Métricas y clasificación multiclase con `diagnostico`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Cargar datos
df = pd.read_csv('mening missing 12.csv')
df.columns = df.columns.str.strip()

if 'diagnostico' in df.columns:
    target_column = 'diagnostico'
elif 'Diagnosis' in df.columns:
    target_column = 'Diagnosis'
else:
    raise KeyError('No se encontró la columna objetivo esperada: diagnostico o Diagnosis')

if target_column != 'diagnostico':
    df = df.rename(columns={target_column: 'diagnostico'})

print('Usando columna objetivo:', 'diagnostico')
print('')
print('Valores de target:')
print(df['diagnostico'].value_counts(dropna=False))

display(df.head())


In [ ]:
# Revisar valores faltantes
df_missing = df.isna().sum().sort_values(ascending=False)
if df_missing.iloc[0] > 0:
    display(df_missing[df_missing > 0])
else:
    print('No hay valores faltantes.')


In [ ]:
# Preparar X e y
drop_cols = ['Patient_ID']
drop_cols = [c for c in drop_cols if c in df.columns]
X = df.drop(columns=['diagnostico'] + drop_cols)
y = df['diagnostico'].astype('string').str.strip()
y = y.replace('', pd.NA)
# Quitar filas sin valor en la variable objetivo
mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
print('Features numéricas:', numeric_features)
print('Features categóricas:', cat_features)

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False))])

preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, cat_features)])

pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline.fit(X_train, y_train)


In [ ]:
# Evaluación de modelo
y_pred = pipeline.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('')
print('Reporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y), ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Verdadero')
ax.set_title('Matriz de confusión')
plt.tight_layout()
plt.show()


In [ ]:
# Distribución de clases en el conjunto de prueba
print(y_test.value_counts())
